#### **What can we do to improvise the previous architecture?**

The previous architecture we designed can track only upto one object in an image what if we had more than one object? 

The primary limitation of a standard global regression architecture is its inability to detect multiple objects simultaneously, as it is mathematically constrained to output a single set of coordinates per image.

To resolve this, we can discretize the input image into a localized grid of smaller, uniform sub-regions (or 'cells'). By shifting the network's objective from a single global prediction to multiple localized predictions, each individual cell becomes responsible for detecting at most one object whose center falls within its boundaries. If a sub-region contains no object, it is simply classified as background. This grid-based approach effectively transforms a complex, multi-object detection task into parallel, single-object regression problems.

<img src = "https://hvp.44e.myftpupload.com/wp-content/uploads/2026/01/image-34-1024x687.jpeg">

#### **How do we create Grid Division?**

How Convolutional Layers Enable Multi-Object DetectionUsing a Convolutional Neural Network (CNN) for object detection allows us to predict a specific number of variables per grid cell. The total number of predicted variables equals:$$\text{Total Variables} = 1 \text{ (Object Confidence)} + 4 \text{ (Bounding Box Coordinates)} + N \text{ (Number of Classes)}$$Instead of predicting one global set of coordinates, the convolutional backbone processes the image while maintaining spatial conservation, effectively treating the output as a grid of smaller localized images. This allows us to construct an output tensor of shape:$$\mathbf{(S, S, \text{Object Confidence} + 4 + N)}$$

In this architecture, the channels inside each individual $(x, y)$ grid cell represent the object's presence, its local coordinates, its global dimensions, and its classification probabilities.
Channel Mapping Example
For a model trained to detect 6 specific types of bone fractures, the channel breakdown for every single cell in the $S \times S$ grid maps out like this:
- Channel 0 $\to$ Object Confidence: The probability that an object's center falls inside this specific grid cell (bounded between 0.0 and 1.0).
- Channel 1 $\to$ Box Center X ($x_c$): The horizontal center of the bounding box, calculated relative to the boundaries of the current cell.
- Channel 2 $\to$ Box Center Y ($y_c$): The vertical center of the bounding box, calculated relative to the boundaries of the current cell.
- Channel 3 $\to$ Box Width ($w$): The total width of the bounding box, scaled relative to the dimensions of the entire image.
- Channel 4 $\to$ Box Height ($h$): The total height of the bounding box, scaled relative to the dimensions of the entire image.
- Channel 5 $\to$ Probability: Elbow Fracture
- Channel 6 $\to$ Probability: Fingers Fracture
- Channel 7 $\to$ Probability: Forearm Fracture
- Channel 8 $\to$ Probability: Humerus Fracture
- Channel 9 $\to$ Probability: Shoulder Fracture
- Channel 10 $\to$ Probability: Wrist Fracture

By organizing the channels this way, the final $1 \times 1$ convolutional layer acts as an array of parallel detectors. If multiple fractures exist across different regions of the image, the corresponding spatial cells will activate and output their respective coordinates simultaneously without interfering with one another.

#### **NOTE : Convolutional layer preserve spactial data**

So the above method is similar to creating grids

#### **You must have noted that we predict box center why do we do that?**

We have different objects in our image but image a grid cell that has 50% of class 0 and 50% of class 1 what will the grid cell be forced to predict? Also note that we predict Box width and Box Height relative to the size of image so forcing a grid cell to predict center is more practical then forcing indivisual cells to predict indivisually

Um! We are close to learn how we use YOLO models now 

The box width and height are predicted using sigmoid .. listening to the word sigmoid you must have noted that this can cause vansishing gradients problem because the $$ \text{derivative of sigmoid(x)} = \text{sigmoid(x)(1 - sigmoid(x))}$$ The maximum of this function can be 0.25. So this causes the vanishing gradient problem in the architecture.

#### **ANCHOR BOXES**

**Anchor Boxes (Handling Scale Diversity)**

If a single cell is responsible for a giant object, how does it know how to calculate those massive shapes cleanly without its gradients exploding or vanishing? It uses Anchor Boxes (or prior boxes).

Instead of making the network guess box dimensions completely from scratch, engineers look at the training dataset beforehand and calculate the most common object shapes (e.g., small square boxes for finger fractures, tall skinny boxes for forearm fractures, and giant wide boxes for shoulders).

Every grid cell is given a set of these pre-defined shapes as templates:
- **Anchor 1:** Small Square (Fingers)
- **Anchor 2:** Medium Vertical Rectangle (Forearm)
- **Anchor 3:** Large Horizontal Rectangle (Shoulder/Humerus)

Instead of predicting raw sizes, the cell simply predicts a scaling factor to tweak the closest-matching anchor template:

$$ \text{Final Width} = \text{Anchor Width} \times e^{\text{predicted scale}} $$

If a massive object is present, the cell naturally selects its largest anchor template and scales it up slightly, allowing it to easily capture objects much larger than the cell itself.

### Non-Maximum Suppression (NMS)

Because large objects cover so much territory, a side effect occurs: multiple adjacent grid cells might get confused and all attempt to predict a bounding box for the same giant object. To clean up this mess, **Non-Maximum Suppression (NMS)** filters the final outputs at the very end of the pipeline.



NMS operates through the following algorithmic steps:

1. **Identify the Highest Confidence Box:** It reviews all predicted boxes across the entire grid and selects the one with the absolute highest **Object Confidence** score.
2. **Measure the Overlap (IoU):** It calculates how much the other neighboring boxes overlap with this top-performing box using a metric called **Intersection over Union (IoU)**:

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

3. **Suppress the Duplicates:** If a nearby box overlaps with the best box by more than a pre-defined threshold (e.g., more than $50\%$ overlap), NMS assumes they are targeting the same object and aggressively deletes the weaker box.
4. **Repeat:** This loop repeats for the remaining boxes until no overlapping duplicates are left.

This process eliminates the clutter, leaving you with exactly one clean, perfectly fitted bounding box around the massive fracture.

Now lets see how to build these models simply

In [1]:
# Import required modules

from ultralytics import YOLO


In [2]:
model = YOLO("yolov8n.pt")

In [ ]:
results = model.train(
    data = "./dataset/data.yaml",
    epochs = 13,
    imgsz = 1024 
    #note that I am passing this argument because ultralytics defaults these to 640px but for suppose
    #you have an image of size 1280 and the default argument this image is compressed automatically to 640px which can cause the
    #loss of spacial information to your model similar for an image of smaller dimension
)

Ultralytics 8.4.60 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 5806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=13, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma